**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Sparse Coding & Dictionary Learning

[Compressed Sensing](./Compressed_Sensing.ipynb) assumed a known sparsifying basis. This sequel asks two harder questions: how do you find the sparse code *greedily and fast* (OMP), and — the big one — can you **learn the dictionary itself from data** (K-SVD)? Both verified on planted-truth problems where we know the answer.

## 1. Pre-requisites

[Compressed Sensing](./Compressed_Sensing.ipynb), [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2/S4.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Greedy Pursuit: OMP* (~40 min)
**Goal:** build the sparse code one atom at a time; verify exact recovery of a planted support.
**Builds on:** [Compressed Sensing](./Compressed_Sensing.ipynb). &nbsp; **Feeds into:** Session 2 (dictionary learning).

---

## 2. One Atom at a Time

💡 **Intuition.** L1 minimization is principled but iterative-solver-shaped. **Orthogonal Matching Pursuit** is the greedy engineer's answer: repeatedly pick the atom most correlated with the residual, then re-fit *all* chosen atoms by least squares (the 'orthogonal' — the residual stays perpendicular to everything chosen, so no atom is picked twice). $K$ iterations, each a correlation + a small solve; exact recovery when the dictionary is incoherent enough.

In [2]:
def omp(D, y, K):
    resid = y.copy(); support = []
    for _ in range(K):
        support.append(int(np.argmax(np.abs(D.T @ resid))))
        Ds = D[:, support]
        coef, *_ = np.linalg.lstsq(Ds, y, rcond=None)
        resid = y - Ds @ coef
    x = np.zeros(D.shape[1]); x[support] = coef
    return x, sorted(support)

# ORACLE: planted 6-sparse code in a random 64×256 dictionary — recover it exactly
n_dim, n_atoms, K = 64, 256, 6
D = rng.standard_normal((n_dim, n_atoms)); D /= np.linalg.norm(D, axis=0)
true_supp = sorted(rng.choice(n_atoms, K, replace=False).tolist())
x_true = np.zeros(n_atoms); x_true[true_supp] = rng.standard_normal(K) * 2
y = D @ x_true

x_hat, supp = omp(D, y, K)
print(f"true support:      {true_supp}")
print(f"OMP support:       {supp}")
print(f"coefficient error: {np.abs(x_hat - x_true).max():.2e}")
assert supp == true_supp

true support:      [11, 33, 150, 214, 234, 238]
OMP support:       [11, 33, 150, 214, 234, 238]
coefficient error: 2.44e-15


In [3]:
# and its breaking point: recovery probability vs sparsity level (the coherence wall)
def trial(K_s):
    ts = sorted(rng.choice(n_atoms, K_s, replace=False).tolist())
    x = np.zeros(n_atoms); x[ts] = rng.standard_normal(K_s)
    _, s = omp(D, D @ x, K_s)
    return s == ts
Ks = [4, 8, 12, 16, 20, 24, 28]
rates = [np.mean([trial(K_s) for _ in range(150)]) for K_s in Ks]
plt.figure(figsize=(7, 2.6))
plt.plot(Ks, rates, "o-")
plt.xlabel("sparsity K"); plt.ylabel("exact recovery rate")
plt.title(f"64 measurements, 256 atoms: greedy recovery falls off a cliff")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

/tmp/ipykernel_2983624/2767358051.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *K-SVD: Learning the Dictionary* (~40 min)
**Goal:** alternate sparse coding and per-atom SVD updates; recover a PLANTED dictionary.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (denoising with learned atoms).

---

## 3. Where Do Atoms Come From?

💡 **Intuition.** Wavelets are a guess; data can vote. **K-SVD** alternates: (1) sparse-code every training signal with the current dictionary (OMP), (2) update each atom — restrict to the signals that *use* it, and set the atom (and its coefficients) to the **rank-one SVD** of their residual matrix — the best single direction explaining what's left ([Eckart–Young](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) again). The audit most demos skip: plant a dictionary, generate data from it, and count how many atoms K-SVD *actually recovers*.

In [4]:
def ksvd(Y, n_atoms, K_s, n_iter=30):
    D = Y[:, rng.choice(Y.shape[1], n_atoms, replace=False)].astype(float)
    D /= np.linalg.norm(D, axis=0)
    for it in range(n_iter):
        X = np.stack([omp(D, y, K_s)[0] for y in Y.T], axis=1)      # sparse coding
        for j in range(n_atoms):                                     # atom-by-atom update
            users = np.nonzero(X[j])[0]
            if len(users) == 0:
                D[:, j] = rng.standard_normal(D.shape[0]); D[:, j] /= np.linalg.norm(D[:, j]); continue
            E = Y[:, users] - D @ X[:, users] + np.outer(D[:, j], X[j, users])
            U, s, Vt = np.linalg.svd(E, full_matrices=False)
            D[:, j] = U[:, 0]; X[j, users] = s[0] * Vt[0]
    return D, X

# ORACLE: plant a 20-atom dictionary in R^16, generate 3-sparse data, recover the atoms
n_dim2, n_at2, K2 = 16, 20, 3
D_true = rng.standard_normal((n_dim2, n_at2)); D_true /= np.linalg.norm(D_true, axis=0)
Y = np.stack([D_true[:, rng.choice(n_at2, K2, replace=False)] @ rng.standard_normal(K2)
              for _ in range(3000)], axis=1)
Y += 0.01 * rng.standard_normal(Y.shape)

D_learn, _ = ksvd(Y, n_at2, K2, n_iter=45)
# match learned atoms to true atoms (up to sign and permutation)
sims = np.abs(D_learn.T @ D_true)
recovered = int((sims.max(0) > 0.98).sum())
print(f"planted atoms recovered (|cos| > 0.98): {recovered} / {n_at2}")
print("(with ample data + iterations: all 20; starve either and K-SVD drops atoms to local minima)")
assert recovered >= 16

planted atoms recovered (|cos| > 0.98): 20 / 20
(with ample data + iterations: all 20; starve either and K-SVD drops atoms to local minima)


---
### 🕐 Session 3 of 3 — *Denoising with Learned Atoms* (~35 min)
**Goal:** the payoff: a dictionary learned from noisy patches beats a fixed basis at denoising.
**Builds on:** Session 2.

---

## 4. The Payoff

💡 **Intuition.** Denoise by *sparse approximation*: code each noisy patch with a few atoms, rebuild — whatever the dictionary can express survives, and noise (which is sparse in **no** dictionary) dies. Learned atoms fit the data's actual structure better than any fixed basis fits it, so at equal sparsity they keep more signal. This pipeline (K-SVD denoising) was state-of-the-art for a decade and is the conceptual ancestor of every learned-representation denoiser since — including [diffusion models](../Intro_Mach_Learn/Diffusion_Models.ipynb).

In [5]:
# 1-D 'patches' from a piecewise-smooth signal family; compare DCT vs learned dictionary
from scipy.fft import dct
def make_patch():
    t = np.linspace(0, 1, n_dim2)
    kind = rng.integers(3)
    if kind == 0:  return np.sin(2*np.pi*rng.uniform(1, 3)*t + rng.uniform(0, 6))
    if kind == 1:  return np.sign(t - rng.uniform(0.2, 0.8)) * rng.uniform(0.5, 1)
    return np.exp(-((t - rng.uniform(0.2, 0.8))/0.08)**2)

train = np.stack([make_patch() for _ in range(2000)], axis=1)
D_data, _ = ksvd(train + 0.02*rng.standard_normal(train.shape), 24, 3, n_iter=20)
D_dct = dct(np.eye(n_dim2), norm="ortho", axis=0)          # full 16-atom DCT basis

# denoise fresh noisy patches by 3-sparse OMP approximation in each dictionary
sigma_n = 0.25
test = np.stack([make_patch() for _ in range(300)], axis=1)
noisy = test + sigma_n * rng.standard_normal(test.shape)
def denoise(D_use):
    out = np.stack([D_use @ omp(D_use, y, 3)[0] for y in noisy.T], axis=1)
    return 10*np.log10(np.var(noisy - test) / np.var(out - test))
print(f"input SNR {10*np.log10(np.var(test)/sigma_n**2):.1f} dB")
print(f"denoising gain — DCT basis: {denoise(D_dct):+.1f} dB   learned dictionary: {denoise(D_data):+.1f} dB")

input SNR 7.8 dB
denoising gain — DCT basis: -0.8 dB   learned dictionary: +2.5 dB


## 5. Conclusion

OMP builds codes greedily and provably recovers planted supports; K-SVD recovers planted *dictionaries* atom-for-atom; and sparse approximation in the learned dictionary is a denoiser that knows your data. Representation is no longer a design choice — it's a fit.

---
## Where next

- [Compressed Sensing](./Compressed_Sensing.ipynb) — the convex counterpart.
- [Representation Learning](../Intro_Mach_Learn/Representation_Learning.ipynb) — the neural descendants of this exact idea.